In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [21]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [22]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [23]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3009862005710602
epoch:  1, loss: 0.17274008691310883
epoch:  2, loss: 0.1052151620388031
epoch:  3, loss: 0.07207281142473221
epoch:  4, loss: 0.05587145313620567
epoch:  5, loss: 0.048011187463998795
epoch:  6, loss: 0.044232938438653946
epoch:  7, loss: 0.0424315445125103
epoch:  8, loss: 0.04157819598913193
epoch:  9, loss: 0.041175320744514465
epoch:  10, loss: 0.04098512977361679
epoch:  11, loss: 0.04089483246207237
epoch:  12, loss: 0.04085131734609604
epoch:  13, loss: 0.04082966223359108
epoch:  14, loss: 0.04081820324063301
epoch:  15, loss: 0.04081153869628906
epoch:  16, loss: 0.04080016911029816
epoch:  17, loss: 0.04078841954469681
epoch:  18, loss: 0.040781568735837936
epoch:  19, loss: 0.040769267827272415
epoch:  20, loss: 0.04075704514980316
epoch:  21, loss: 0.04074989631772041
epoch:  22, loss: 0.04073674604296684
epoch:  23, loss: 0.0407242476940155
epoch:  24, loss: 0.0407169908285141
epoch:  25, loss: 0.04070398211479187
epoch:  26, loss: 0.04

In [24]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7417227625846863
Test metrics:  R2 = 0.7597920894622803


In [25]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="scaled",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.323582261800766
epoch:  1, loss: 0.19601024687290192
epoch:  2, loss: 0.08703628182411194
epoch:  3, loss: 0.05848308280110359
epoch:  4, loss: 0.046869538724422455
epoch:  5, loss: 0.041045188903808594
epoch:  6, loss: 0.04091963171958923
epoch:  7, loss: 0.04081578552722931
epoch:  8, loss: 0.04080357402563095
epoch:  9, loss: 0.04079432040452957
epoch:  10, loss: 0.04079311713576317
epoch:  11, loss: 0.04079180210828781
epoch:  12, loss: 0.04079049453139305
epoch:  13, loss: 0.04078923910856247
epoch:  14, loss: 0.04078804701566696
epoch:  15, loss: 0.040786903351545334
epoch:  16, loss: 0.04078581556677818
epoch:  17, loss: 0.04078477621078491
epoch:  18, loss: 0.04078378155827522
epoch:  19, loss: 0.040782827883958817
epoch:  20, loss: 0.04078191518783569
epoch:  21, loss: 0.04078104719519615
epoch:  22, loss: 0.04078022018074989
epoch:  23, loss: 0.04077942669391632
epoch:  24, loss: 0.04077867418527603
epoch:  25, loss: 0.04077795520424843
epoch:  26, loss: 0.

In [26]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -5436.87744140625
Test metrics:  R2 = -4890.00830078125


In [27]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="BB1",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.06662198901176453
epoch:  1, loss: 0.0557141974568367
epoch:  2, loss: 0.04942842200398445
epoch:  3, loss: 0.04582124203443527
epoch:  4, loss: 0.043758656829595566
epoch:  5, loss: 0.042582519352436066
epoch:  6, loss: 0.04191276431083679
epoch:  7, loss: 0.04153120517730713
epoch:  8, loss: 0.04131319001317024
epoch:  9, loss: 0.04118776321411133
epoch:  10, loss: 0.041114650666713715
epoch:  11, loss: 0.04107103869318962
epoch:  12, loss: 0.04104408621788025
epoch:  13, loss: 0.041025303304195404
epoch:  14, loss: 0.040992848575115204
epoch:  15, loss: 0.04098564013838768
epoch:  16, loss: 0.04094601050019264
epoch:  17, loss: 0.04092150181531906
epoch:  18, loss: 0.040902938693761826
epoch:  19, loss: 0.04087325930595398
epoch:  20, loss: 0.040864285081624985
epoch:  21, loss: 0.040827978402376175
epoch:  22, loss: 0.04080534726381302
epoch:  23, loss: 0.040784768760204315
epoch:  24, loss: 0.04075728356838226
epoch:  25, loss: 0.040742892771959305
epoch:  26, l

In [28]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7254461646080017
Test metrics:  R2 = 0.7444156408309937


In [29]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="BB2",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.09571767598390579
epoch:  1, loss: 0.06916981190443039
epoch:  2, loss: 0.055649321526288986
epoch:  3, loss: 0.04836767539381981
epoch:  4, loss: 0.04441482946276665
epoch:  5, loss: 0.04227042198181152
epoch:  6, loss: 0.041103072464466095
epoch:  7, loss: 0.04046161472797394
epoch:  8, loss: 0.04010064899921417
epoch:  9, loss: 0.03988528996706009
epoch:  10, loss: 0.0397537425160408
epoch:  11, loss: 0.03946889191865921
epoch:  12, loss: 0.03931929171085358
epoch:  13, loss: 0.03923623263835907
epoch:  14, loss: 0.03918619453907013
epoch:  15, loss: 0.039103277027606964
epoch:  16, loss: 0.03901413083076477
epoch:  17, loss: 0.03896176069974899
epoch:  18, loss: 0.038880329579114914
epoch:  19, loss: 0.038790278136730194
epoch:  20, loss: 0.03873812407255173
epoch:  21, loss: 0.038664765655994415
epoch:  22, loss: 0.038569312542676926
epoch:  23, loss: 0.038514405488967896
epoch:  24, loss: 0.03843941166996956
epoch:  25, loss: 0.038334548473358154
epoch:  26, lo

In [30]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8106637001037598
Test metrics:  R2 = 0.8122259378433228


In [31]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="quadratic",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2683975100517273
epoch:  1, loss: 0.17001856863498688
epoch:  2, loss: 0.0870886966586113
epoch:  3, loss: 0.0466865710914135
epoch:  4, loss: 0.04078328609466553
epoch:  5, loss: 0.04060995951294899
epoch:  6, loss: 0.04058023542165756
epoch:  7, loss: 0.04056946560740471
epoch:  8, loss: 0.040550388395786285
epoch:  9, loss: 0.04053119197487831
epoch:  10, loss: 0.04051179438829422
epoch:  11, loss: 0.040492210537195206
epoch:  12, loss: 0.04047249257564545
epoch:  13, loss: 0.04045228287577629
epoch:  14, loss: 0.040432777255773544
epoch:  15, loss: 0.04041435196995735
epoch:  16, loss: 0.04039906710386276
epoch:  17, loss: 0.04037603735923767
epoch:  18, loss: 0.04036122187972069
epoch:  19, loss: 0.04034136235713959
epoch:  20, loss: 0.04031934589147568
epoch:  21, loss: 0.0402977280318737
epoch:  22, loss: 0.04027627781033516
epoch:  23, loss: 0.040255241096019745
epoch:  24, loss: 0.040236905217170715
epoch:  25, loss: 0.04021375626325607
epoch:  26, loss: 0.0

In [32]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7362121343612671
Test metrics:  R2 = 0.7656217813491821


In [33]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="lipschitz",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.6055493950843811
epoch:  1, loss: 0.33894702792167664
epoch:  2, loss: 0.19057650864124298
epoch:  3, loss: 0.1101827621459961
epoch:  4, loss: 0.07000851631164551
epoch:  5, loss: 0.052019547671079636
epoch:  6, loss: 0.044590506702661514
epoch:  7, loss: 0.0417051687836647
epoch:  8, loss: 0.04062779247760773
epoch:  9, loss: 0.04022730886936188
epoch:  10, loss: 0.04007063806056976
epoch:  11, loss: 0.039999570697546005
epoch:  12, loss: 0.039958011358976364
epoch:  13, loss: 0.03977201506495476
epoch:  14, loss: 0.03969378024339676
epoch:  15, loss: 0.03968273475766182
epoch:  16, loss: 0.03947838395833969
epoch:  17, loss: 0.039392102509737015
epoch:  18, loss: 0.03934627026319504
epoch:  19, loss: 0.039192333817481995
epoch:  20, loss: 0.03909244388341904
epoch:  21, loss: 0.039042841643095016
epoch:  22, loss: 0.038900136947631836
epoch:  23, loss: 0.03878994286060333
epoch:  24, loss: 0.03873593360185623
epoch:  25, loss: 0.038608990609645844
epoch:  26, loss

In [34]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7965999841690063
Test metrics:  R2 = 0.7941095232963562


In [35]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr_init=1,
    lr_method="keep",
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3433815836906433
epoch:  1, loss: 0.1977313905954361
epoch:  2, loss: 0.12420378625392914
epoch:  3, loss: 0.08564145863056183
epoch:  4, loss: 0.06555241346359253
epoch:  5, loss: 0.05526832491159439
epoch:  6, loss: 0.049435634166002274
epoch:  7, loss: 0.04593779519200325
epoch:  8, loss: 0.043802667409181595
epoch:  9, loss: 0.042489662766456604
epoch:  10, loss: 0.04168088734149933
epoch:  11, loss: 0.04118030518293381
epoch:  12, loss: 0.040869731456041336
epoch:  13, loss: 0.040677111595869064
epoch:  14, loss: 0.04055662453174591
epoch:  15, loss: 0.04048101231455803
epoch:  16, loss: 0.04043259471654892
epoch:  17, loss: 0.04040149226784706
epoch:  18, loss: 0.04038137197494507
epoch:  19, loss: 0.04036777839064598
epoch:  20, loss: 0.040358152240514755
epoch:  21, loss: 0.04035087674856186
epoch:  22, loss: 0.04034528508782387
epoch:  23, loss: 0.040340784937143326
epoch:  24, loss: 0.040336985141038895
epoch:  25, loss: 0.04033348336815834
epoch:  26, loss

In [36]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7150826454162598
Test metrics:  R2 = 0.734723687171936
